# Synthetic vs. Real Spam Signal Comparison

Does our generator produce realistic spam?

We compare spam signal rates (URLs, ALL CAPS, currency symbols etc.) between:
- **Real data**: profiles computed from the actual benchmark dataset
- **Synthetic data**: generated samples from the latest GET pipeline run

## Setup

In [ ]:
import sys
import os
import json
import glob

project_root = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root: {project_root}")

## Load Real Data Signals

From the profile JSON generated by `python -m framework.profile_dataset --task spam`.

In [ ]:
profile_path = os.path.join(project_root, "framework", "data", "profiles", "spam_profile.json")

with open(profile_path, encoding="utf-8") as f:
    profile = json.load(f)

real_signals = profile["spam_signals"]

print(f"Real data — {profile['num_samples']} samples")
print(f"Label distribution: {profile['label_distribution']}")
print()
for label, stats in real_signals.items():
    print(f"{label}: {stats}")

## Load Synthetic Data & Compute Signals

We reuse `analyze_spam_signals()` from the profiler on the generated samples.

In [ ]:
from framework.profiling.spam_profiler import analyze_spam_signals

generated_dir = os.path.join(project_root, "framework", "data", "generated", "spam")
all_files     = sorted(glob.glob(os.path.join(generated_dir, "*.json")))
latest_session = sorted(set(os.path.basename(f)[:15] for f in all_files))[-1]
session_files  = [f for f in all_files if os.path.basename(f).startswith(latest_session)]

print(f"Session: {latest_session}")
print(f"Files:   {[os.path.basename(f) for f in session_files]}")

# Pool all runs and map to {text, label} format
synthetic_rows = []
for path in session_files:
    with open(path, encoding="utf-8") as f:
        for item in json.load(f):
            label = "HAM" if item["error_type"] == "paraphrase" else "SPAM"
            synthetic_rows.append({"text": item["corrupted"], "label": label})

synthetic_signals = analyze_spam_signals(synthetic_rows)

print(f"\nSynthetic data — {len(synthetic_rows)} samples")
for label, stats in synthetic_signals.items():
    print(f"{label}: {stats}")

## Comparison

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

signals    = ["url_rate", "currency_rate", "caps_rate", "exclaim_rate", "keyword_rate"]
signal_labels = ["URL", "Currency", "ALL CAPS", "Exclamation", "Spam Keywords"]
labels_to_plot = [l for l in ["SPAM", "HAM"] if l in real_signals and l in synthetic_signals]

fig, axes = plt.subplots(1, len(labels_to_plot), figsize=(14, 5), sharey=False)
if len(labels_to_plot) == 1:
    axes = [axes]

x     = np.arange(len(signals))
width = 0.35

for ax, label in zip(axes, labels_to_plot):
    real_vals      = [real_signals[label][s]      for s in signals]
    synthetic_vals = [synthetic_signals[label][s] for s in signals]

    ax.bar(x - width/2, real_vals,      width, label="real",      color="#3498db")
    ax.bar(x + width/2, synthetic_vals, width, label="synthetic", color="#3498db", hatch="///", alpha=0.6)

    ax.set_title(f"{label} messages")
    ax.set_xticks(x)
    ax.set_xticklabels(signal_labels, rotation=15, ha="right", fontsize=9)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Fraction of texts")
    ax.legend()
    ax.yaxis.grid(True, linestyle="--", alpha=0.7)

fig.suptitle("Spam Signal Rates — Real vs. Synthetic", fontsize=13)
fig.tight_layout()
plt.savefig("synthetic_vs_real_signals.png", dpi=150)
plt.show()
print("Plot saved to synthetic_vs_real_signals.png")

## Interpretation

If synthetic signal rates are **close to real**: the generator produces realistic spam — good for evaluation validity.

If synthetic rates are **higher than real** (e.g. caps_rate): the generator over-exaggerates spam features, making spam too easy to detect → inflated synthetic scores.

If synthetic rates are **lower than real**: the generator produces too subtle spam — models may not be sufficiently challenged.